# Model D: fast prefix-LM pretraining for downstream phases


## Configuration


In [73]:
from pathlib import Path
import os, sys, json, time, hashlib, gc

ROOT = Path('/kaggle/working/cs781-model-d-fast')
RUN_NAME = 'model_d_prefixlm_82m_fast250m'
RUN_TRAINING = True

TRAIN_RAW_TOKEN_BUDGET = 200000000
TRAIN_BLOCK_LENGTH = 512
TRAIN_PREFIX_LENGTH = 256
RAW_TOKENS_PER_UPDATE = 65_536
MICROBATCH_CANDIDATES = [16, 8, 4, 2, 1]   # Per GPU; capacity checks only, no optimizer trial runs.
MAX_GPUS = 2
MAX_SESSION_MINUTES = None               # Optional graceful stop with a resumable checkpoint.

LOCAL_TOKEN_STORE = None                 # Directory containing token_manifest.json and all .bin files.
TOKEN_STORE_REPO = 'I-230482/cs781-phase1-tokenstore'
ALLOW_TOKEN_DOWNLOAD = True
VERIFY_TOKEN_HASHES = False              # Manifest + exact file sizes always checked; full hashes optional.
RESTORE_RUN_ROOT = None                  # e.g. '/kaggle/input/my-d-output/model_d_resume.zip'
BUILD_RESUME_ARCHIVE = True              # Saves an exact-resume bundle as well as the smaller Phase 2 bundle.

# Keep the original architecture and embedding/tokenizer contract.
MODEL_CONFIG = dict(vocab_size=32000, context_length=2048, train_prefix_length=1024,
                    d_model=640, n_layers=12, n_heads=10, d_ff=1792,
                    dropout=0.10, norm_epsilon=1e-5, tie_embeddings=True)
TRAINING_CONFIG = dict(
    block_length=TRAIN_BLOCK_LENGTH, prefix_length=TRAIN_PREFIX_LENGTH,
    train_raw_token_budget=TRAIN_RAW_TOKEN_BUDGET,
    raw_tokens_per_optimizer_step=RAW_TOKENS_PER_UPDATE,
    microbatch_candidates=MICROBATCH_CANDIDATES,
    learning_rate=2e-4, weight_decay=0.1, warmup_ratio=0.02,
    betas=[0.9, 0.95], epsilon=1e-8, max_grad_norm=1.0,
    seed=42, num_workers=0, cpu_threads=2, memory_headroom_gib=1.5,
    logging_steps=10, eval_steps=200, checkpoint_steps=100,
    first_checkpoint_step=10, keep_checkpoints=2,
    interim_validation_raw_tokens=131_072, validation_raw_tokens=262_144,
)
ROOT.mkdir(parents=True, exist_ok=True)
RUNTIME = ROOT / 'runtime'
RUNTIME.mkdir(exist_ok=True)
RUN_DIR = ROOT / RUN_NAME
TOKENIZER_PATH = ROOT / 'tokenizer'
TOKENIZER_PATH.mkdir(exist_ok=True)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print('Project:', ROOT)
print('Training enabled:', RUN_TRAINING)


Project: /kaggle/working/cs781-model-d-fast
Training enabled: True


## Dependencies and device


In [74]:
import importlib.util, subprocess

if importlib.util.find_spec('torch') is None:
    raise RuntimeError('Use a Kaggle GPU notebook with PyTorch already installed.')
needed = []
for package, requirement in [('tokenizers', 'tokenizers==0.21.2'),
                             ('huggingface_hub', 'huggingface_hub>=0.34,<1')]:
    if importlib.util.find_spec(package) is None:
        needed.append(requirement)
if needed:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *needed])

import torch
torch.set_num_threads(2)
WORLD_SIZE = min(MAX_GPUS, torch.cuda.device_count())
if RUN_TRAINING:
    if not torch.cuda.is_available() or WORLD_SIZE < 1:
        raise RuntimeError('Enable Kaggle GPU T4 x2 (or one available GPU), then Run All.')
    for index in range(WORLD_SIZE):
        props = torch.cuda.get_device_properties(index)
        print('GPU', index, props.name, round(props.total_memory / 2**30, 2), 'GiB')
print('PyTorch:', torch.__version__, '| GPU workers:', WORLD_SIZE)


GPU 0 Tesla T4 14.56 GiB
GPU 1 Tesla T4 14.56 GiB
PyTorch: 2.11.0+cu128 | GPU workers: 2


## Model and training modules


In [75]:
os.chdir(RUNTIME)
if str(RUNTIME) not in sys.path:
    sys.path.insert(0, str(RUNTIME))


In [76]:
%%writefile legal_prefix_model.py
from __future__ import annotations
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


def parameter_counts(model):
    seen = set()
    total = trainable = 0
    for p in model.parameters():
        if id(p) in seen:
            continue
        seen.add(id(p))
        total += p.numel()
        if p.requires_grad:
            trainable += p.numel()
    return {"total": total, "trainable": trainable}


def sinusoidal_positions(max_len, dim):
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(
        torch.arange(0, dim, 2, dtype=torch.float32)
        * (-math.log(10000.0) / dim)
    )
    table = torch.zeros(max_len, dim, dtype=torch.float32)
    table[:, 0::2] = torch.sin(pos * div)
    table[:, 1::2] = torch.cos(pos * div)
    return table


def prefix_lm_mask(q_len, k_len, past_len, prefix_length, device):
    """Boolean SDPA mask: True means this query/key pair may attend."""
    q = torch.arange(past_len, past_len + q_len, device=device)[:, None]
    k = torch.arange(k_len, device=device)[None, :]
    prefix_q = q < int(prefix_length)
    return torch.where(prefix_q, k < int(prefix_length), k <= q)


class PrefixSelfAttention(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D, H = int(cfg["d_model"]), int(cfg["n_heads"])
        if D % H:
            raise ValueError("d_model must be divisible by n_heads")
        self.d_model = D
        self.n_heads = H
        self.head_dim = D // H
        self.dropout_p = float(cfg["dropout"])
        self.qkv = nn.Linear(D, 3 * D, bias=True)
        self.out = nn.Linear(D, D, bias=True)
        self.resid_dropout = nn.Dropout(self.dropout_p)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat((pk, k), dim=-2)
            v = torch.cat((pv, v), dim=-2)

        present = (k, v) if use_cache else None
        a = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,
        )
        a = a.transpose(1, 2).contiguous().view(B, T, D)
        return self.resid_dropout(self.out(a)), present


class GEGLU(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D, Ff = int(cfg["d_model"]), int(cfg["d_ff"])
        self.gate = nn.Linear(D, Ff, bias=True)
        self.up = nn.Linear(D, Ff, bias=True)
        self.down = nn.Linear(Ff, D, bias=True)
        self.dropout = nn.Dropout(float(cfg["dropout"]))

    def forward(self, x):
        return self.dropout(
            self.down(F.gelu(self.gate(x), approximate="tanh") * self.up(x))
        )


class PrefixBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        D, eps = int(cfg["d_model"]), float(cfg["norm_epsilon"])
        self.norm1 = nn.LayerNorm(D, eps=eps)
        self.attn = PrefixSelfAttention(cfg)
        self.norm2 = nn.LayerNorm(D, eps=eps)
        self.ffn = GEGLU(cfg)

    def forward(self, x, attn_mask=None, past_kv=None, use_cache=False):
        a, present = self.attn(
            self.norm1(x), attn_mask=attn_mask, past_kv=past_kv, use_cache=use_cache
        )
        x = x + a
        x = x + self.ffn(self.norm2(x))
        return x, present


class PrefixLMLegalModel(nn.Module):
    """Single-stack hybrid Prefix-LM / UniLM-style legal language model.

    Prefix positions attend bidirectionally inside the prefix. Suffix positions attend
    to the whole prefix plus their causal suffix history. Training scores only suffix
    targets; generation treats the entire official prompt as the bidirectional prefix.
    """

    def __init__(self, cfg):
        super().__init__()
        self.cfg = dict(cfg)
        required = (
            "vocab_size", "context_length", "train_prefix_length", "d_model",
            "n_layers", "n_heads", "d_ff", "dropout", "norm_epsilon",
            "tie_embeddings",
        )
        for key in required:
            if key not in cfg:
                raise ValueError(f"Missing config field: {key}")

        C, P = int(cfg["context_length"]), int(cfg["train_prefix_length"])
        D, H = int(cfg["d_model"]), int(cfg["n_heads"])
        if not (1 <= P < C):
            raise ValueError("train_prefix_length must be inside context")
        if D % H:
            raise ValueError("d_model must be divisible by n_heads")

        self.token_embedding = nn.Embedding(int(cfg["vocab_size"]), D)
        self.register_buffer(
            "position_table", sinusoidal_positions(C, D), persistent=False
        )
        # Precompute the only large mask used in production training.
        self.register_buffer(
            "train_attention_mask",
            prefix_lm_mask(C, C, 0, P, torch.device("cpu")),
            persistent=False,
        )
        self.embedding_dropout = nn.Dropout(float(cfg["dropout"]))
        self.blocks = nn.ModuleList(
            [PrefixBlock(cfg) for _ in range(int(cfg["n_layers"]))]
        )
        self.final_norm = nn.LayerNorm(D, eps=float(cfg["norm_epsilon"]))
        self.lm_head = nn.Linear(D, int(cfg["vocab_size"]), bias=False)

        self.apply(self._init_weights)
        residual_scale = 1.0 / math.sqrt(2.0 * int(cfg["n_layers"]))
        for block in self.blocks:
            block.attn.out.weight.data.mul_(residual_scale)
            block.ffn.down.weight.data.mul_(residual_scale)

        if bool(cfg["tie_embeddings"]):
            self.lm_head.weight = self.token_embedding.weight

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(
        self,
        input_ids,
        prefix_length=None,
        past_key_values=None,
        use_cache=False,
        loss_only=False,
    ):
        if input_ids.ndim != 2:
            raise ValueError("input_ids must be [B,T]")
        _, T = input_ids.shape
        if T <= 0:
            raise ValueError("Empty input")

        if past_key_values is None:
            past_key_values = [None] * len(self.blocks)
            past_len = 0
        else:
            if len(past_key_values) != len(self.blocks):
                raise ValueError("Bad KV cache length")
            past_len = (
                0 if past_key_values[0] is None
                else int(past_key_values[0][0].shape[-2])
            )

        total_len = past_len + T
        if total_len > int(self.cfg["context_length"]):
            raise ValueError("Context length exceeded")

        if prefix_length is None:
            prefix_length = int(self.cfg["train_prefix_length"])
        prefix_length = int(prefix_length)
        if not (1 <= prefix_length <= total_len):
            raise ValueError("Invalid prefix_length")
        if loss_only:
            if past_len:
                raise ValueError("loss_only does not support cached input")
            if prefix_length >= T:
                raise ValueError("Need suffix target tokens for Prefix-LM loss")

        tok = self.token_embedding(input_ids)
        pos = self.position_table[past_len:total_len].to(
            device=input_ids.device, dtype=tok.dtype
        )
        x = self.embedding_dropout(tok + pos.unsqueeze(0))

        # Build/share one attention mask for all blocks. During one-token cached
        # generation the query is a suffix position and every cached key is legal.
        if past_len and T == 1 and past_len >= prefix_length:
            attn_mask = None
        elif (
            past_len == 0
            and T == int(self.cfg["context_length"])
            and prefix_length == int(self.cfg["train_prefix_length"])
        ):
            attn_mask = self.train_attention_mask
        elif past_len == 0 and prefix_length == T:
            # Entire sequence is the bidirectional prompt prefix.
            attn_mask = None
        else:
            attn_mask = prefix_lm_mask(
                T, total_len, past_len, prefix_length, input_ids.device
            )

        presents = []
        for block, pkv in zip(self.blocks, past_key_values):
            x, present = block(
                x, attn_mask=attn_mask, past_kv=pkv, use_cache=use_cache
            )
            if use_cache:
                presents.append(present)

        x = self.final_norm(x)

        if loss_only:
            # Prefix raw positions: [0, P-1]. Score raw targets [P, T-1].
            # Logit at P-1 predicts first suffix token P. Project only the
            # supervised half, which substantially reduces training memory.
            start = prefix_length - 1
            pred_hidden = x[:, start:-1, :].contiguous()
            target = input_ids[:, prefix_length:].contiguous()
            if pred_hidden.shape[:2] != target.shape:
                raise RuntimeError("Prefix-LM target shape mismatch")
            pred = self.lm_head(pred_hidden)
            loss = F.cross_entropy(
                pred.view(-1, pred.shape[-1]), target.view(-1)
            )
            # Length-1 tensors gather cleanly under DataParallel.
            return loss.view(1)

        logits = self.lm_head(x)
        return (logits, presents) if use_cache else logits


Overwriting legal_prefix_model.py


In [77]:
%%writefile legal_prefix_data.py
from __future__ import annotations
import bisect, hashlib, json, math, random
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, Sampler


def require(c, m):
    if not c:
        raise ValueError(m)


def json_hash(obj):
    return hashlib.sha256(
        json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":")).encode("utf-8")
    ).hexdigest()


def file_hash(path, block_size=8 * 1024 * 1024):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for b in iter(lambda: f.read(block_size), b""):
            h.update(b)
    return h.hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def tokenizer_identity(path):
    path = Path(path)
    names = ("tokenizer.json", "tokenizer_config.json", "special_tokens_map.json")
    hashes = {n: file_hash(path / n) for n in names if (path / n).is_file()}
    require("tokenizer.json" in hashes, "Missing tokenizer.json")
    return {"files": hashes, "sha256": json_hash(hashes)}


class PackedDataset(Dataset):
    """Memory-map complete context-length blocks from the existing token streams."""
    def __init__(self, directory, split, context_length=2048, verify_hashes=False):
        self.root = Path(directory).resolve()
        self.manifest = read_json(self.root / "token_manifest.json")
        check = dict(self.manifest)
        checksum = check.pop("manifest_hash")
        require(json_hash(check) == checksum, "Token manifest checksum mismatch")
        require(split in ("train", "validation"), "Invalid split")
        require(int(context_length) >= 2, "Context length must be >=2")
        self.context_length = int(context_length)
        self.dtype = np.dtype(self.manifest["dtype"])
        self.streams, self.ends, self.maps = [], [], {}
        total = 0
        for stream in self.manifest["streams"]:
            if stream["split"] != split:
                continue
            path = self.root / stream["file"]
            require(path.is_file(), f"Missing token stream: {path}")
            require(path.stat().st_size == int(stream["bytes"]), f"Size mismatch: {path.name}")
            if verify_hashes:
                require(file_hash(path) == stream["sha256"], f"Hash mismatch: {path.name}")
            blocks = int(stream["tokens"]) // self.context_length
            if blocks:
                self.streams.append(stream)
                total += blocks
                self.ends.append(total)
        require(total > 0, f"No complete {split} blocks")

    def __len__(self):
        return self.ends[-1]

    def __getitem__(self, index):
        if index < 0:
            index += len(self)
        require(0 <= index < len(self), "Block index out of range")
        fi = bisect.bisect_right(self.ends, index)
        prior = self.ends[fi - 1] if fi else 0
        if fi not in self.maps:
            self.maps[fi] = np.memmap(
                self.root / self.streams[fi]["file"], dtype=self.dtype, mode="r"
            )
        start = (index - prior) * self.context_length
        view = self.maps[fi][start:start + self.context_length]
        ids = torch.from_numpy(np.asarray(view, dtype=np.int64).copy())
        require(ids.numel() == self.context_length, "Short packed block")
        return {"input_ids": ids}

    def __getstate__(self):
        state = self.__dict__.copy()
        state["maps"] = {}
        return state

    def close(self):
        for m in self.maps.values():
            try:
                m._mmap.close()
            except Exception:
                pass
        self.maps.clear()

    def __del__(self):
        try:
            self.close()
        except Exception:
            pass


class ResumableBatches(Sampler):
    """Deterministic affine permutation; start_microbatch makes resume exact."""
    def __init__(self, size, batch_size, start_microbatch, microbatches, seed):
        self.size = int(size)
        self.batch_size = int(batch_size)
        self.start = int(start_microbatch)
        self.count = int(microbatches)
        self.seed = int(seed)
        require(self.size > 0 and self.batch_size > 0 and self.count >= 0, "Invalid sampler")

    def __len__(self):
        return self.count

    def __iter__(self):
        epoch_cache = None
        a, b = 1, 0
        for micro in range(self.start, self.start + self.count):
            indices = []
            for absolute in range(micro * self.batch_size, (micro + 1) * self.batch_size):
                epoch, offset = divmod(absolute, self.size)
                if epoch != epoch_cache:
                    rng = random.Random(f"{self.seed}:block-order:{epoch}")
                    a = rng.randrange(1, self.size + 1)
                    while math.gcd(a, self.size) != 1:
                        a = a % self.size + 1
                    b = rng.randrange(self.size)
                    epoch_cache = epoch
                indices.append((a * offset + b) % self.size)
            yield indices


class EvenlySpacedSubset(Dataset):
    def __init__(self, dataset, count):
        self.dataset = dataset
        self.count = int(count)
    def __len__(self):
        return self.count
    def __getitem__(self, index):
        return self.dataset[index * len(self.dataset) // self.count]


def seed_worker(worker_id):
    seed = torch.initial_seed() % 2**32
    np.random.seed(seed)
    random.seed(seed)


def loader_options(cfg):
    workers = int(cfg["num_workers"])
    out = {
        "num_workers": workers,
        "pin_memory": bool(cfg["pin_memory"]),
        "worker_init_fn": seed_worker,
        "generator": torch.Generator().manual_seed(int(cfg["seed"])),
    }
    if workers:
        out.update(
            persistent_workers=bool(cfg["persistent_workers"]),
            prefetch_factor=int(cfg["prefetch_factor"]),
        )
    return out


Overwriting legal_prefix_data.py


In [78]:
%%writefile model_d_fast_train.py
"""Fast training glue for the unchanged Model D architecture from MODEL D.ipynb.

No pretrained weights, architectural rewrite, or parameter search. Runtime-only
microbatch capacity checks preserve the global token batch. CPU runs are explicit
software fixtures and their checkpoints are rejected by the Phase 2 production
loader.
"""
from __future__ import annotations

import argparse
import contextlib
import datetime
import gc
import json
import math
import os
import random
import time
from pathlib import Path

import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, Subset

import legal_prefix_data as data
from legal_prefix_model import PrefixLMLegalModel, parameter_counts


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("w", encoding="utf-8") as stream:
        json.dump(value, stream, indent=2, ensure_ascii=False, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def atomic_save(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".partial")
    with temporary.open("wb") as stream:
        torch.save(value, stream)
        stream.flush()
        os.fsync(stream.fileno())
    temporary.replace(path)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def rng_state(device):
    return {"python": random.getstate(), "numpy": np.random.get_state(),
            "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state(device) if device.type == "cuda" else None}


def restore_rng(state, device):
    random.setstate(state["python"])
    np.random.set_state(state["numpy"])
    torch.set_rng_state(state["torch"])
    if state["cuda"] is not None:
        torch.cuda.set_rng_state(state["cuda"], device)


def native_bf16(device):
    if device.type != "cuda":
        return False
    with torch.cuda.device(device):
        try:
            return torch.cuda.is_bf16_supported(including_emulation=False)
        except TypeError:
            return torch.cuda.get_device_capability(device)[0] >= 8


def amp(device, precision):
    if precision == "fp32":
        return contextlib.nullcontext()
    return torch.autocast(device_type="cuda",
                          dtype=torch.bfloat16 if precision == "bf16" else torch.float16)


def plan(model_config, training, world, microbatch=None):
    block = int(training["block_length"])
    prefix = int(training["prefix_length"])
    raw_per_step = int(training["raw_tokens_per_optimizer_step"])
    if not 1 <= prefix < block <= int(model_config["context_length"]):
        raise ValueError("Training prefix/window exceeds the model's context capacity.")
    if raw_per_step % block:
        raise ValueError("The global raw-token batch must divide into complete windows.")
    global_windows = raw_per_step // block
    if global_windows % world:
        raise ValueError("Global windows must divide equally between GPU processes.")
    microbatch = int(microbatch or training["microbatch_candidates"][0])
    if global_windows % (world * microbatch):
        raise ValueError("Per-GPU microbatch must divide the global window batch.")
    steps = int(training["train_raw_token_budget"]) // raw_per_step
    if steps < 1:
        raise ValueError("Budget must cover at least one global optimizer batch.")
    return {"steps": steps, "block_length": block, "prefix_length": prefix,
            "world_size": world, "microbatch_per_gpu": microbatch,
            "global_windows_per_step": global_windows,
            "accumulation_steps": global_windows // (world * microbatch),
            "raw_tokens_per_step": raw_per_step,
            "scored_tokens_per_step": global_windows * (block - prefix),
            "requested_raw_tokens": int(training["train_raw_token_budget"]),
            "planned_raw_tokens": steps * raw_per_step,
            "planned_scored_tokens": steps * global_windows * (block - prefix),
            "unused_budget_tokens": int(training["train_raw_token_budget"]) % raw_per_step}


class RankBatches(data.ResumableBatches):
    """The original deterministic global sampler, sliced into disjoint ranks."""
    def __init__(self, size, per_gpu, rank, world, start, count, seed):
        super().__init__(size, per_gpu * world, start, count, seed)
        self.rank, self.per_gpu = rank, per_gpu

    def __iter__(self):
        for indices in super().__iter__():
            first = self.rank * self.per_gpu
            yield indices[first:first + self.per_gpu]


def loader_options(training, device):
    workers = int(training["num_workers"])
    options = {"num_workers": workers, "pin_memory": device.type == "cuda",
               "generator": torch.Generator().manual_seed(int(training["seed"]))}
    if workers:
        options.update(persistent_workers=True, prefetch_factor=2,
                       multiprocessing_context="spawn", worker_init_fn=data.seed_worker)
    return options


def check_capacity(model, request, device, precision, world):
    """No optimizer steps; reserve Adam moments and leave DDP/allocator headroom."""
    training = request["training"]
    before = rng_state(device)
    choice = None
    try:
        for candidate in training["microbatch_candidates"]:
            try:
                geometry = plan(request["model_config"], training, world, candidate)
            except ValueError:
                continue
            ids = loss = reserve = None
            success = 1
            peak = 0.0
            try:
                torch.cuda.empty_cache()
                torch.cuda.reset_peak_memory_stats(device)
                # AdamW creates two FP32 moment arrays after the first update.
                reserve = [torch.empty_like(p) for p in model.parameters() for _ in range(2)]
                ids = torch.randint(4, request["model_config"]["vocab_size"],
                                    (candidate, geometry["block_length"]), device=device)
                model.train()
                with amp(device, precision):
                    loss = model(ids, prefix_length=geometry["prefix_length"], loss_only=True).mean()
                loss.backward()
                torch.cuda.synchronize(device)
                if not torch.isfinite(loss).item():
                    raise RuntimeError("Capacity probe produced nonfinite loss.")
                peak = torch.cuda.max_memory_reserved(device) / 2**30
                capacity = torch.cuda.get_device_properties(device).total_memory / 2**30
                if peak > capacity - float(training["memory_headroom_gib"]):
                    success = 0
            except torch.cuda.OutOfMemoryError:
                success = 0
            finally:
                model.zero_grad(set_to_none=True)
                ids = loss = reserve = None
                gc.collect()
                torch.cuda.empty_cache()
            flag = torch.tensor(success, dtype=torch.int32, device=device)
            memory = torch.tensor(peak, dtype=torch.float64, device=device)
            if world > 1:
                dist.all_reduce(flag, op=dist.ReduceOp.MIN)
                dist.all_reduce(memory, op=dist.ReduceOp.MAX)
            if int(os.environ.get("RANK", 0)) == 0:
                print("CAPACITY", {"microbatch_per_gpu": candidate,
                      "fits_with_headroom": bool(flag.item()),
                      "max_reserved_gib": round(memory.item(), 3),
                      "optimizer_updates": 0}, flush=True)
            if flag.item():
                choice = geometry
                break
    finally:
        restore_rng(before, device)
    if choice is None:
        raise RuntimeError("No proposed microbatch fits. Inspect GPU usage and restart the session.")
    return choice


def implementation_identity(request, training_dataset, effective, software_smoke):
    folder = Path(__file__).resolve().parent
    return {"model": dict(request["model_config"]), "training": dict(request["training"]),
            "effective_runtime": effective,
            "dataset_manifest": training_dataset.manifest["manifest_hash"],
            "tokenizer": data.tokenizer_identity(request["tokenizer_path"])["sha256"],
            "implementation": {name: data.file_hash(folder / name) for name in
                                ("legal_prefix_model.py", "legal_prefix_data.py", "model_d_fast_train.py")},
            "software_test_fixture": bool(software_smoke)}


def optimizer_scheduler(model, training, geometry, device):
    decay, no_decay = [], []
    for parameter in model.parameters():
        (decay if parameter.ndim >= 2 else no_decay).append(parameter)
    groups = [{"params": decay, "weight_decay": training["weight_decay"]},
              {"params": no_decay, "weight_decay": 0.0}]
    options = {"lr": training["learning_rate"], "betas": tuple(training["betas"]),
               "eps": training["epsilon"]}
    if device.type == "cuda":
        options["fused"] = True
    try:
        optimizer = torch.optim.AdamW(groups, **options)
    except (TypeError, RuntimeError):
        options.pop("fused", None)
        optimizer = torch.optim.AdamW(groups, **options)
    warmup = max(1, math.ceil(geometry["steps"] * training["warmup_ratio"]))

    def multiplier(step):
        if step < warmup:
            return (step + 1) / warmup
        progress = min(1.0, (step - warmup) / max(1, geometry["steps"] - warmup))
        return 0.5 * (1 + math.cos(math.pi * progress))

    return optimizer, torch.optim.lr_scheduler.LambdaLR(optimizer, multiplier)


@torch.inference_mode()
def validate(raw, dataset, geometry, training, device, precision, rank, world, final=False):
    budget = training["validation_raw_tokens"] if final else training["interim_validation_raw_tokens"]
    count = min(len(dataset), int(budget) // geometry["block_length"])
    if count < 1:
        raise ValueError("Validation needs at least one complete window.")
    # No duplicate padding and no discarded examples, even for unequal rank counts.
    indices = [(i * len(dataset)) // count for i in range(rank, count, world)]
    loader = DataLoader(Subset(dataset, indices), batch_size=geometry["microbatch_per_gpu"],
                        **loader_options(training, device))
    was_training = raw.training
    raw.eval()
    stats = torch.zeros(2, dtype=torch.float64, device=device)
    try:
        for batch in loader:
            ids = batch["input_ids"].to(device, non_blocking=True)
            with amp(device, precision):
                loss = raw(ids, prefix_length=geometry["prefix_length"], loss_only=True).mean()
            targets = ids.shape[0] * (geometry["block_length"] - geometry["prefix_length"])
            stats[0] += loss.double() * targets
            stats[1] += targets
        if world > 1:
            dist.all_reduce(stats)
    finally:
        raw.train(was_training)
    mean = (stats[0] / stats[1]).item()
    if not math.isfinite(mean):
        raise RuntimeError("Nonfinite validation loss.")
    return {"validation_loss": mean, "validation_perplexity": math.exp(min(mean, 80)),
            "validation_targets": int(stats[1].item()),
            "validation_raw_tokens": count * geometry["block_length"]}


def save_checkpoint(root, raw, optimizer, scheduler, scaler, state, identity,
                    device, rank, world, keep):
    random_states = [rng_state(device)]
    if world > 1:
        random_states = [None] * world
        dist.all_gather_object(random_states, rng_state(device))
    full_path = root / f"checkpoint-{state['step']:08d}.pt"
    compact_folder = root.parent / "phase2_checkpoint"
    compact_path = compact_folder / full_path.name
    if rank == 0:
        shared = {"model": raw.state_dict(), "state": dict(state), "identity": identity,
                  "software_test_fixture": identity["software_test_fixture"]}
        atomic_save(full_path, {**shared, "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(),
                    "rng_per_rank": random_states, "world_size": world})
        if state["optimizer_steps"] > 0:
            atomic_save(compact_path, shared)
        # Prune only after both new files were successfully saved atomically.
        for previous in sorted(root.glob("checkpoint-????????.pt"))[:-int(keep)]:
            previous.unlink()
        if compact_path.exists():
            for previous in compact_folder.glob("checkpoint-????????.pt"):
                if previous != compact_path:
                    previous.unlink()
        write_json(root / "progress.json", state)
        write_json(root / "checkpoint_paths.json", {"resume": str(full_path),
                   "phase2": str(compact_path) if compact_path.exists() else None, "state": state})
        print("CHECKPOINT", {"resume": str(full_path),
              "phase2": str(compact_path) if compact_path.exists() else None}, flush=True)
    if world > 1:
        dist.barrier()
    return full_path, compact_path


def run(request, *, software_smoke=False):
    rank = int(os.environ.get("RANK", 0))
    world = int(os.environ.get("WORLD_SIZE", 1))
    local_rank = int(os.environ.get("LOCAL_RANK", 0))
    device = torch.device("cpu" if software_smoke else f"cuda:{local_rank}")
    if not software_smoke and not torch.cuda.is_available():
        raise RuntimeError("Enable GPU in Kaggle before real training. CPU is reserved for software fixtures.")
    if software_smoke and world != 1:
        raise ValueError("The software smoke runner uses one CPU process.")
    if device.type == "cuda":
        torch.cuda.set_device(device)
    if world > 1 and not dist.is_initialized():
        dist.init_process_group("nccl", timeout=datetime.timedelta(minutes=10), device_id=device)
    torch.set_num_threads(int(request["training"]["cpu_threads"]))
    torch.use_deterministic_algorithms(False)
    seed_all(int(request["training"]["seed"]))
    root = Path(request["run_dir"])
    root.mkdir(parents=True, exist_ok=True)
    stop_path = root / "STOP_AFTER_CURRENT_UPDATE"
    if rank == 0:
        stop_path.unlink(missing_ok=True)
    if world > 1:
        dist.barrier()
    training = request["training"]
    train_data = data.PackedDataset(request["token_store"], "train", training["block_length"])
    val_data = data.PackedDataset(request["token_store"], "validation", training["block_length"])
    raw = PrefixLMLegalModel(request["model_config"]).to(device)
    count = parameter_counts(raw)["total"]
    if not software_smoke and not 50_000_000 <= count <= 300_000_000:
        raise ValueError("Model D must remain inside the capstone's parameter range.")
    precision = "fp32" if software_smoke else "bf16" if native_bf16(device) else "fp16"
    if world > 1:
        native = torch.tensor(int(precision == "bf16"), device=device)
        dist.all_reduce(native, op=dist.ReduceOp.MIN)
        precision = "bf16" if native.item() else "fp16"
    checkpoints = sorted(root.glob("checkpoint-????????.pt"))
    saved = torch.load(checkpoints[-1], map_location="cpu", weights_only=False) if checkpoints else None
    if saved is not None:
        if "rng_per_rank" not in saved:
            raise RuntimeError("Resume requires this fast trainer's full checkpoint, not a Phase 2 weights export.")
        previous_runtime = saved["identity"]["effective_runtime"]
        geometry = plan(request["model_config"], training, world,
                        previous_runtime["microbatch_per_gpu"])
        effective = {**geometry, "precision": precision}
    else:
        if (root / "training.jsonl").exists():
            raise RuntimeError("A log exists without a complete checkpoint. Use a fresh RUN_NAME to start again.")
        geometry = (plan(request["model_config"], training, world) if software_smoke else
                    check_capacity(raw, request, device, precision, world))
        effective = {**geometry, "precision": precision}
    identity = implementation_identity(request, train_data, effective, software_smoke)
    if saved is not None and saved["identity"] != identity:
        raise RuntimeError("Resume config, GPU count, precision, source, tokenizer, or data changed; use a fresh run.")
    if geometry["planned_raw_tokens"] > len(train_data) * geometry["block_length"]:
        raise ValueError("Token store has insufficient unique training windows for this budget.")
    optimizer, scheduler = optimizer_scheduler(raw, training, geometry, device)
    scaler = torch.amp.GradScaler("cuda", enabled=precision == "fp16", init_scale=1024.0)
    state = {"step": 0, "optimizer_steps": 0, "raw_tokens": 0, "scored_tokens": 0,
             "optimizer_raw_tokens": 0, "optimizer_scored_tokens": 0,
             "amp_skipped_updates": 0, "cursor": 0, "elapsed": 0.0, "validation": {}}
    if saved is not None:
        raw.load_state_dict(saved["model"], strict=True)
        optimizer.load_state_dict(saved["optimizer"])
        scheduler.load_state_dict(saved["scheduler"])
        scaler.load_state_dict(saved["scaler"])
        state = saved["state"]
    if (state["raw_tokens"] != state["step"] * geometry["raw_tokens_per_step"] or
        state["scored_tokens"] != state["step"] * geometry["scored_tokens_per_step"] or
        state["cursor"] != state["step"] * geometry["accumulation_steps"]):
        raise RuntimeError("Checkpoint token/cursor counters do not match the run plan.")
    if rank == 0:
        write_json(root / "training_manifest.json", {"identity": identity,
                   "parameters": count, "plan": geometry,
                   "budget_definition": "Complete raw input windows; suffix targets are counted separately. AMP overflow attempts consume the raw budget."})
        print("MODEL D", count, "parameters", json.dumps(effective), flush=True)
    if state["step"] == geometry["steps"]:
        if rank == 0:
            print("Completed run already exists; no training repeated.", flush=True)
        return state
    if state["step"] > geometry["steps"]:
        raise RuntimeError("Saved step exceeds the current training budget.")
    model = DDP(raw, device_ids=[local_rank], broadcast_buffers=False,
                gradient_as_bucket_view=True) if world > 1 else raw
    sampler = RankBatches(len(train_data), geometry["microbatch_per_gpu"], rank, world,
                          state["cursor"], (geometry["steps"] - state["step"]) * geometry["accumulation_steps"],
                          int(training["seed"]))
    batches = iter(DataLoader(train_data, batch_sampler=sampler,
                              **loader_options(training, device)))
    if saved is not None:
        restore_rng(saved["rng_per_rank"][rank], device)
    else:
        seed_all(int(training["seed"]) + rank)
    saved = None
    gc.collect()
    session_start = time.perf_counter()
    prior_elapsed = state["elapsed"]
    session_start_tokens = state["raw_tokens"]
    window_start = session_start
    window_raw_tokens = state["raw_tokens"]
    raw.train()
    log = (root / "training.jsonl").open("a", encoding="utf-8", buffering=1) if rank == 0 else None
    try:
        while state["step"] < geometry["steps"]:
            optimizer.zero_grad(set_to_none=True)
            local_loss = torch.zeros((), dtype=torch.float64, device=device)
            learning_rate = optimizer.param_groups[0]["lr"]
            for micro in range(geometry["accumulation_steps"]):
                ids = next(batches)["input_ids"].to(device, non_blocking=True)
                sync = (model.no_sync() if world > 1 and micro + 1 < geometry["accumulation_steps"]
                        else contextlib.nullcontext())
                # DDP no_sync must cover both the forward and backward passes.
                with sync:
                    with amp(device, precision):
                        loss = model(ids, prefix_length=geometry["prefix_length"], loss_only=True).mean()
                    local_loss += loss.detach().double()
                    scaler.scale(loss / geometry["accumulation_steps"]).backward()
            scaler.unscale_(optimizer)
            norm = torch.nn.utils.clip_grad_norm_(raw.parameters(), training["max_grad_norm"],
                                                  error_if_nonfinite=False)
            finite_loss = torch.isfinite(local_loss).to(torch.int32)
            if world > 1:
                dist.all_reduce(finite_loss, op=dist.ReduceOp.MIN)
            if not finite_loss.item():
                raise RuntimeError("Nonfinite training loss; inspect the last complete checkpoint.")
            if precision != "fp16" and not torch.isfinite(norm).item():
                raise RuntimeError("Nonfinite gradients in a run without FP16 loss scaling.")
            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            applied = scaler.get_scale() >= scale_before
            if applied:
                scheduler.step()
                state["optimizer_steps"] += 1
                state["optimizer_raw_tokens"] += geometry["raw_tokens_per_step"]
                state["optimizer_scored_tokens"] += geometry["scored_tokens_per_step"]
            else:
                state["amp_skipped_updates"] += 1
            state["step"] += 1
            state["cursor"] += geometry["accumulation_steps"]
            state["raw_tokens"] += geometry["raw_tokens_per_step"]
            state["scored_tokens"] += geometry["scored_tokens_per_step"]
            reached = state["step"] == geometry["steps"]
            stop = 0
            if rank == 0:
                limit = request.get("max_session_minutes")
                stop = int(stop_path.exists() or
                           (limit is not None and time.perf_counter() - session_start >= float(limit) * 60) or
                           state["amp_skipped_updates"] > 50)
            stop_tensor = torch.tensor(stop, dtype=torch.int32, device=device)
            if world > 1:
                dist.broadcast(stop_tensor, src=0)
            stopped = bool(stop_tensor.item())
            if reached or state["step"] % int(training["eval_steps"]) == 0:
                state["validation"] = validate(raw, val_data, geometry, training, device, precision,
                                               rank, world, final=reached)
                if rank == 0:
                    print("VALIDATION", state["validation"], flush=True)
            elapsed_session = time.perf_counter() - session_start
            state["elapsed"] = prior_elapsed + elapsed_session
            should_log = (state["step"] == 1 or state["step"] % int(training["logging_steps"]) == 0 or
                          reached or stopped)
            if should_log:
                if world > 1:
                    dist.all_reduce(local_loss)
                if rank == 0:
                    now = time.perf_counter()
                    rate = (state["raw_tokens"] - window_raw_tokens) / max(now - window_start, 1e-9)
                    average_rate = (state["raw_tokens"] - session_start_tokens) / max(elapsed_session, 1e-9)
                    row = {"step": state["step"], "raw_tokens": state["raw_tokens"],
                           "scored_tokens": state["scored_tokens"], "optimizer_steps": state["optimizer_steps"],
                           "amp_skipped_updates": state["amp_skipped_updates"],
                           "loss": local_loss.item() / (world * geometry["accumulation_steps"]),
                           "learning_rate": learning_rate,
                           "gradient_norm": norm.item() if torch.isfinite(norm).item() else None,
                           "elapsed_seconds": state["elapsed"], "recent_raw_tokens_per_second": rate,
                           "average_raw_tokens_per_second": average_rate,
                           "estimated_remaining_training_minutes": (
                               (geometry["planned_raw_tokens"] - state["raw_tokens"]) / max(average_rate, 1e-9) / 60
                               if state["step"] >= int(training["first_checkpoint_step"]) else None)}
                    log.write(json.dumps(row, allow_nan=False) + "\n")
                    print("TRAIN", json.dumps(row), flush=True)
                    window_start, window_raw_tokens = now, state["raw_tokens"]
            should_save = (reached or stopped or state["step"] == int(training["first_checkpoint_step"]) or
                           state["step"] % int(training["checkpoint_steps"]) == 0)
            if should_save:
                save_checkpoint(root, raw, optimizer, scheduler, scaler, state, identity,
                                device, rank, world, training["keep_checkpoints"])
            if stopped:
                break
        if state["optimizer_steps"] == 0:
            raise RuntimeError("No optimizer updates were applied; this is not a trained checkpoint.")
        if rank == 0:
            write_json(root / "training_summary.json", {**state,
                       "target_reached": state["step"] == geometry["steps"],
                       "requested_raw_tokens": geometry["requested_raw_tokens"],
                       "planned_raw_tokens": geometry["planned_raw_tokens"],
                       "session_seconds": time.perf_counter() - session_start,
                       "parameters": count, "software_test_fixture": software_smoke})
            print("DONE", {"step": state["step"], "target_reached": state["step"] == geometry["steps"],
                  "raw_tokens": state["raw_tokens"], "scored_tokens": state["scored_tokens"]}, flush=True)
    finally:
        if log is not None:
            log.close()
        train_data.close()
        val_data.close()
    return state


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    arguments = parser.parse_args()
    request = json.loads(Path(arguments.config).read_text())
    try:
        run(request)
    finally:
        if dist.is_initialized():
            dist.destroy_process_group()


if __name__ == "__main__":
    main()


Overwriting model_d_fast_train.py


In [79]:
%%writefile model_d_fast_utils.py
"""Input staging, safe restore and checkpoint export for the fast Model D notebook."""
from __future__ import annotations
import json
import shutil
import zipfile
from pathlib import Path
import torch
from huggingface_hub import HfApi, hf_hub_download, snapshot_download
import legal_prefix_data as data
from model_d_fast_train import write_json, atomic_save


EXPECTED_TOKENIZER_SHA256 = "3163063597ba6358484fc37200c76f548040a2a58186862b519904ba427deb2b"
EXPECTED_MANIFEST_HASH = "a9eaf6d0c2c1d99d5f83ce8506fdc08cd729908556eae796217f7c8e8c0e638f"


def restore_project(source, project):
    if source is None:
        return
    source, project = Path(source).resolve(), Path(project).resolve()
    if not source.exists():
        raise FileNotFoundError(source)
    if any(project.rglob("checkpoint-????????.pt")):
        print("Local checkpoints already exist; supplied restore source was not copied again.")
        return
    project.mkdir(parents=True, exist_ok=True)
    if source.is_dir():
        if not (source / "fast_run_request.json").is_file():
            raise ValueError("RESTORE_RUN_ROOT must be the saved project root containing fast_run_request.json.")
        if source == project or project.is_relative_to(source):
            raise ValueError("Restore source and destination must be separate directories.")
        shutil.copytree(source, project, dirs_exist_ok=True)
    else:
        with zipfile.ZipFile(source) as archive:
            for entry in archive.infolist():
                destination = (project / entry.filename).resolve()
                if not destination.is_relative_to(project):
                    raise ValueError("Unsafe archive member path.")
            if "fast_run_request.json" not in archive.namelist():
                raise ValueError("Use model_d_resume.zip, not the Phase 2 weights-only archive.")
            archive.extractall(project)
    print("Restored saved fast Model D project to", project)


def discover_token_store(override, download_folder):
    if override is not None:
        folder = Path(override)
        if not (folder / "token_manifest.json").is_file():
            raise FileNotFoundError("LOCAL_TOKEN_STORE needs the directory containing token_manifest.json.")
        return folder
    default = Path("/kaggle/working/cs781-phase1-tokenstore")
    candidates = [p for p in [Path(download_folder), default] if (p / "token_manifest.json").is_file()]
    if candidates:
        return candidates[0]
    input_folder = Path("/kaggle/input")
    attached = list(input_folder.rglob("token_manifest.json")) if input_folder.exists() else []
    matching = []
    for manifest_path in attached:
        try:
            manifest = data.read_json(manifest_path)
            if manifest.get("manifest_hash") == EXPECTED_MANIFEST_HASH:
                matching.append(manifest_path.parent)
        except (ValueError, OSError):
            continue
    if len(matching) > 1:
        raise RuntimeError("Several matching token stores are attached; set LOCAL_TOKEN_STORE explicitly.")
    return matching[0] if matching else Path(download_folder)


def prepare_token_store(folder, token, repo, *, allow_download=True, verify_hashes=False):
    folder = Path(folder)
    manifest_path = folder / "token_manifest.json"
    revision = None
    if not manifest_path.is_file():
        if not allow_download:
            raise FileNotFoundError("Attach your original token store or allow its download.")
        if not token:
            raise RuntimeError("Enable Kaggle Secret HF_TOKEN with read access to your private token-store repo.")
        info = HfApi(token=token).repo_info(repo, repo_type="dataset")
        revision = info.sha
        folder.mkdir(parents=True, exist_ok=True)
        hf_hub_download(repo, "token_manifest.json", repo_type="dataset", revision=revision,
                        token=token, local_dir=str(folder))
    manifest = data.read_json(manifest_path)
    check = dict(manifest)
    recorded_hash = check.pop("manifest_hash", None)
    if recorded_hash != data.json_hash(check) or recorded_hash != EXPECTED_MANIFEST_HASH:
        raise ValueError("This is not the exact, verified packed token store from your original Model D notebook.")
    if int(manifest["vocab_size"]) != 32000:
        raise ValueError("The packed corpus does not use the expected 32k tokenizer.")
    token_identity = manifest.get("identity", {}).get("tokenizer", {}).get("sha256")
    if token_identity is not None and token_identity != EXPECTED_TOKENIZER_SHA256:
        raise ValueError("Token-store/tokenizer provenance mismatch.")
    wanted = [s for s in manifest["streams"] if s["split"] in ("train", "validation")]
    missing = [s["file"] for s in wanted if not (folder / s["file"]).is_file()]
    if missing:
        if not allow_download or not token:
            raise FileNotFoundError("Token stream files are missing. Enable HF_TOKEN/downloads or attach the complete token store.")
        if revision is None:
            revision = HfApi(token=token).repo_info(repo, repo_type="dataset").sha
            # Ensure local manifest and the pinned remote revision refer to the same bytes.
            remote_manifest = hf_hub_download(repo, "token_manifest.json", repo_type="dataset",
                                              revision=revision, token=token)
            if data.file_hash(remote_manifest) != data.file_hash(manifest_path):
                raise RuntimeError("The local and remote token-store manifests differ; select a complete local store.")
        snapshot_download(repo, repo_type="dataset", revision=revision, token=token,
                          local_dir=str(folder), allow_patterns=missing, max_workers=4)
    for stream in wanted:
        path = folder / stream["file"]
        if path.stat().st_size != int(stream["bytes"]):
            raise ValueError("Token stream size mismatch: " + path.name)
        if verify_hashes and data.file_hash(path) != stream["sha256"]:
            raise ValueError("Token stream checksum mismatch: " + path.name)
    print("Packed corpus ready:", folder)
    print("Download uses existing packed tokens; no tokenizer fitting or raw-corpus processing.")
    print("Token-stream bytes:", sum(int(s["bytes"]) for s in wanted))
    return manifest


def make_archive(target, members):
    target = Path(target)
    temporary = target.with_suffix(target.suffix + ".partial")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path, name in members:
            archive.write(path, name)
    temporary.replace(target)
    return target


def export_checkpoints(project, run_dir, expected_tokenizer_sha=EXPECTED_TOKENIZER_SHA256,
                       *, build_resume=True):
    project, run_dir = Path(project), Path(run_dir)
    full = sorted(run_dir.glob("checkpoint-????????.pt"))
    if not full:
        raise FileNotFoundError("No trained Model D checkpoint was saved. Inspect the training error first.")
    latest = full[-1]
    checkpoint = torch.load(latest, map_location="cpu", weights_only=False, mmap=True)
    if checkpoint.get("software_test_fixture"):
        raise ValueError("Software fixture checkpoints cannot be exported as a real Model D.")
    if checkpoint["identity"]["tokenizer"] != expected_tokenizer_sha:
        raise ValueError("Export tokenizer provenance mismatch.")
    if checkpoint["state"]["optimizer_steps"] < 1:
        raise ValueError("No real optimizer update was saved.")
    compact = project / "phase2_checkpoint" / latest.name
    # A completed run is also recoverable if only its full checkpoint was retained.
    if not compact.exists():
        atomic_save(compact, {key: checkpoint[key] for key in
                             ("model", "state", "identity", "software_test_fixture")})
    summary = {"checkpoint": str(compact), "resume_checkpoint": str(latest),
               "raw_tokens": checkpoint["state"]["raw_tokens"],
               "scored_tokens": checkpoint["state"]["scored_tokens"],
               "optimizer_steps": checkpoint["state"]["optimizer_steps"],
               "target_reached": checkpoint["state"]["step"] == checkpoint["identity"]["effective_runtime"]["steps"],
               "tokenizer_sha256": checkpoint["identity"]["tokenizer"],
               "phase2_checkpoint_sha256": data.file_hash(compact)}
    del checkpoint
    write_json(project / "model_d_export.json", summary)
    tokenizer_files = [(p, "tokenizer/" + p.name) for p in sorted((project / "tokenizer").glob("*.json"))]
    small_members = [(compact, "phase2_checkpoint/" + compact.name), *tokenizer_files,
                     (project / "runtime" / "legal_prefix_model.py", "legal_prefix_model.py"),
                     (project / "model_d_export.json", "model_d_export.json"),
                     (run_dir / "training_manifest.json", "training_manifest.json")]
    small_archive = make_archive(project.parent / "model_d_phase2.zip", small_members)
    resume_archive = None
    if build_resume:
        resume_members = [(latest, run_dir.name + "/" + latest.name)]
        for folder in [project / "runtime", project / "tokenizer"]:
            resume_members.extend((p, str(p.relative_to(project))) for p in sorted(folder.glob("*")) if p.is_file())
        resume_members.extend((p, str(p.relative_to(project))) for p in sorted(run_dir.glob("*.json*")) if p.is_file())
        resume_members.extend((p, p.name) for p in [project / "fast_run_request.json", project / "model_d_export.json"])
        resume_archive = make_archive(project.parent / "model_d_resume.zip", resume_members)
    print("Trained Model D checkpoint:", compact)
    print("Target reached:", summary["target_reached"], "| raw/scored tokens:",
          summary["raw_tokens"], summary["scored_tokens"])
    print("Phase 2 archive:", small_archive, "| Resume archive:", resume_archive)
    return summary, small_archive, resume_archive


Overwriting model_d_fast_utils.py


In [80]:
EXPECTED_SOURCE_HASHES = {'legal_prefix_model.py': '05ac895255de353dd033cf975ee6c38ab030cecefe1b59554373839e6565f77b', 'legal_prefix_data.py': '2ab24e917630aa4425a9f45dc71335b36544eae0c936920d9d88f72fd14c2e68', 'model_d_fast_train.py': '5dc9b85121f3ca2e6d71a99de1b738d9e9eb9884acec5855c23aa20de8444cf2', 'model_d_fast_utils.py': 'f23678ef01068516baccd8e5cb0b42dfbd00ac99f98a0a486fdcc60ddacf3f95'}
EXPECTED_SOURCE_BYTES = {'legal_prefix_model.py': 9885, 'legal_prefix_data.py': 5881, 'model_d_fast_train.py': 26413, 'model_d_fast_utils.py': 9938}
for filename, expected in EXPECTED_SOURCE_HASHES.items():
    path = RUNTIME / filename
    payload = path.read_bytes()
    # IPython writefile adds one terminal newline to original files that lack it.
    if len(payload) == EXPECTED_SOURCE_BYTES[filename] + 1 and payload.endswith(b'\n'):
        payload = payload[:-1]
        path.write_bytes(payload)
    assert hashlib.sha256(payload).hexdigest() == expected, filename

from model_d_fast_utils import restore_project, discover_token_store, prepare_token_store, export_checkpoints
restore_project(RESTORE_RUN_ROOT, ROOT)
for filename, expected in EXPECTED_SOURCE_HASHES.items():
    assert hashlib.sha256((RUNTIME / filename).read_bytes()).hexdigest() == expected, (
        'Restored runtime differs from this notebook. Resume with the matching notebook version.'
    )
from model_d_fast_train import plan, write_json
from legal_prefix_model import PrefixLMLegalModel, parameter_counts
from legal_prefix_data import tokenizer_identity
with torch.device('meta'):
    shape_model = PrefixLMLegalModel(MODEL_CONFIG)
assert parameter_counts(shape_model)['total'] == 81_541_888
del shape_model
print('Original Model D: 81,541,888 parameters; original source hashes verified.')


Original Model D: 81,541,888 parameters; original source hashes verified.


## Original tokenizer


In [81]:
import base64, zlib
from tokenizers import Tokenizer
_TOKENIZER_BLOBS = {"tokenizer.json": "...", "tokenizer_config.json": "eNrNkz9PwzAQxfdK/Q6RZyRKYUAIMXZm6IaQddjX1KpzjnyOqFrx3bEb558ge6Y499PdPb/kXderohCgNWoZ3AmJpUblNHrxUlwTjHgznOObchSQQqyJ1xr0m7jrkeXgTR3JASzjUCfnK7Dmgvov8zM9bKi0KL+d/6eJa1QGbATBN9iWfzIWD7NyGzotT+52Vi4vT+zjrNj7BalNj1ay+HLc/tgTR4WyCCSbumXmAsE4klyDQp4sETgZ0N9S4Dl4kHl9zk4yJ++tYoasNFQ3QRJUt6kf2cS2ajR3hgkIycckoQI+iVT9HM+p4CwtUhmOccx28/Tcwhi/kbYhjAno6JpkozExi4eQUb4weqkscJIl3j3uPRhCve/gDrhv8A2p7E8e50157HDM1EjDLWHrVfwCv0yuL7c=", "special_tokens_map.json": "eNqr5uVSUFBKyi+OL8nPTs1TslKoBokAxZLz80pS80qAIko2xXZKOlDhnOKSoswCoGhaYk5xKkw0L78oNzEnsyo1BV2mCKv64sy89JzU+PL8IrgGkEwtWF4plYBz9OnsnoLEFLzuAcrT10Gledl4HQSUp5eDeLmACADVA6Ap"}
for name, blob in _TOKENIZER_BLOBS.items():
    (TOKENIZER_PATH / name).write_bytes(zlib.decompress(base64.b64decode(blob)))
TOKENIZER_IDENTITY = tokenizer_identity(TOKENIZER_PATH)
assert TOKENIZER_IDENTITY['sha256'] == '3163063597ba6358484fc37200c76f548040a2a58186862b519904ba427deb2b'
tokenizer = Tokenizer.from_file(str(TOKENIZER_PATH / 'tokenizer.json'))
assert tokenizer.get_vocab_size() == 32000
print('Original 32k tokenizer verified:', TOKENIZER_IDENTITY['sha256'])


Original 32k tokenizer verified: 3163063597ba6358484fc37200c76f548040a2a58186862b519904ba427deb2b


## Architecture checks and training plan


In [82]:
import copy
saved_rng = torch.get_rng_state()
torch.manual_seed(42)
tiny_config = dict(MODEL_CONFIG, vocab_size=64, d_model=32, n_heads=4,
                   n_layers=2, d_ff=64, context_length=32,
                   train_prefix_length=8, dropout=0.0)
tiny = PrefixLMLegalModel(tiny_config).eval()
ids = torch.randint(4, 64, (1, 16))
with torch.no_grad():
    baseline = tiny(ids, prefix_length=8)
    changed = ids.clone()
    changed[:, 8:] = torch.randint(4, 64, (1, 8))
    torch.testing.assert_close(baseline[:, :8], tiny(changed, prefix_length=8)[:, :8])
    changed = ids.clone()
    changed[:, 13:] = torch.randint(4, 64, (1, 3))
    torch.testing.assert_close(baseline[:, :13], tiny(changed, prefix_length=8)[:, :13])
    _, cache = tiny(ids[:, :8], prefix_length=8, use_cache=True)
    cached = []
    for index in range(8, 16):
        logits, cache = tiny(ids[:, index:index+1], prefix_length=8,
                             past_key_values=cache, use_cache=True)
        cached.append(logits)
    torch.testing.assert_close(baseline[:, 8:], torch.cat(cached, dim=1), rtol=2e-4, atol=2e-5)
tiny.train()
tiny(ids, prefix_length=8, loss_only=True).mean().backward()
assert tiny.token_embedding.weight.grad is not None
del tiny, ids, baseline, changed, logits, cache, cached
torch.set_rng_state(saved_rng)

planned = plan(MODEL_CONFIG, TRAINING_CONFIG, max(1, WORLD_SIZE))
assert planned['planned_raw_tokens'] <= TRAIN_RAW_TOKEN_BUDGET
assert planned['unused_budget_tokens'] < RAW_TOKENS_PER_UPDATE
print('CPU prefix, causal suffix, cache, backward: PASS')
print('Desired plan before capacity fallback:', json.dumps(planned, indent=2))


CPU prefix, causal suffix, cache, backward: PASS
Desired plan before capacity fallback: {
  "steps": 3051,
  "block_length": 512,
  "prefix_length": 256,
  "world_size": 2,
  "microbatch_per_gpu": 16,
  "global_windows_per_step": 128,
  "accumulation_steps": 4,
  "raw_tokens_per_step": 65536,
  "scored_tokens_per_step": 32768,
  "requested_raw_tokens": 200000000,
  "planned_raw_tokens": 199950336,
  "planned_scored_tokens": 99975168,
  "unused_budget_tokens": 49664
}


## Packed corpus


In [83]:
TOKEN_STORE = None
if RUN_TRAINING:
    HF_TOKEN = os.environ.get('HF_TOKEN')
    if not HF_TOKEN:
        try:
            from kaggle_secrets import UserSecretsClient
            HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
        except Exception:
            HF_TOKEN = None
    TOKEN_STORE = discover_token_store(LOCAL_TOKEN_STORE, ROOT / 'token_store')
    token_manifest = prepare_token_store(TOKEN_STORE, HF_TOKEN, TOKEN_STORE_REPO,
                                        allow_download=ALLOW_TOKEN_DOWNLOAD,
                                        verify_hashes=VERIFY_TOKEN_HASHES)
else:
    print('RUN_TRAINING=False: data access and download skipped.')


Packed corpus ready: /kaggle/working/cs781-model-d-fast/token_store
Download uses existing packed tokens; no tokenizer fitting or raw-corpus processing.
Token-stream bytes: 410272234


In [84]:
from datetime import datetime, timezone

RESTORE_RUN_ROOT = None
RUN_NAME = "model_d_fresh_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_DIR = ROOT / RUN_NAME

assert not RUN_DIR.exists(), f"Directory already exists: {RUN_DIR}"
print("New training directory:", RUN_DIR)

New training directory: /kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722


In [85]:
trainer = RUNTIME / "model_d_fast_train.py"
source = trainer.read_text()

marker = '    checkpoints = sorted(root.glob("checkpoint-????????.pt"))'
replacement = marker + '''
    print("RESUME_DEBUG", {
        "script": str(Path(__file__).resolve()),
        "run_dir": str(root.resolve()),
        "checkpoints": [str(p) for p in checkpoints],
    }, flush=True)
'''

if "RESUME_DEBUG" not in source:
    assert marker in source, "Trainer differs from the supplied version."
    trainer.write_text(source.replace(marker, replacement, 1))

print("Diagnostic added to:", trainer)

Diagnostic added to: /kaggle/working/cs781-model-d-fast/runtime/model_d_fast_train.py


## Train Model D


In [86]:
TRAINING_SUMMARY = None
if RUN_TRAINING:
    request = dict(model_config=MODEL_CONFIG, training=TRAINING_CONFIG,
                   token_store=str(TOKEN_STORE), tokenizer_path=str(TOKENIZER_PATH),
                   run_dir=str(RUN_DIR), max_session_minutes=MAX_SESSION_MINUTES)
    config_path = ROOT / 'fast_run_request.json'
    write_json(config_path, request)
    command = [sys.executable, '-m', 'torch.distributed.run', '--standalone',
               f'--nproc_per_node={WORLD_SIZE}', str(RUNTIME / 'model_d_fast_train.py'),
               '--config', str(config_path)]
    process = subprocess.Popen(command, cwd=str(RUNTIME),
        env={**os.environ, 'OMP_NUM_THREADS': '2', 'TOKENIZERS_PARALLELISM': 'false'},
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
        start_new_session=True)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    except KeyboardInterrupt:
        RUN_DIR.mkdir(exist_ok=True)
        (RUN_DIR / 'STOP_AFTER_CURRENT_UPDATE').touch()
        print('Stop requested. Trainer is saving after the current update. Wait before restarting.')
        # Return control promptly; rerun the export cell once the trainer reports DONE.
        raise
    if status != 0:
        raise RuntimeError(f'Model D trainer exited with status {status}. See the error above; retain the last checkpoint.')
    summary_path = RUN_DIR / 'training_summary.json'
    if summary_path.exists():
        TRAINING_SUMMARY = json.loads(summary_path.read_text())
        print('Training summary:', json.dumps(TRAINING_SUMMARY, indent=2))
else:
    print('RUN_TRAINING=False: real training was not launched.')


RESUME_DEBUG {'script': '/kaggle/working/cs781-model-d-fast/runtime/model_d_fast_train.py', 'run_dir': '/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722', 'checkpoints': []}
RESUME_DEBUG {'script': '/kaggle/working/cs781-model-d-fast/runtime/model_d_fast_train.py', 'run_dir': '/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722', 'checkpoints': []}
CAPACITY {'microbatch_per_gpu': 16, 'fits_with_headroom': True, 'max_reserved_gib': 5.41, 'optimizer_updates': 0}
MODEL D 81541888 parameters {"steps": 3051, "block_length": 512, "prefix_length": 256, "world_size": 2, "microbatch_per_gpu": 16, "global_windows_per_step": 128, "accumulation_steps": 4, "raw_tokens_per_step": 65536, "scored_tokens_per_step": 32768, "requested_raw_tokens": 200000000, "planned_raw_tokens": 199950336, "planned_scored_tokens": 99975168, "unused_budget_tokens": 49664, "precision": "fp16"}
TRAIN {"step": 1, "raw_tokens": 65536, "scored_tokens": 32768, "optimizer_steps": 1, "amp_skipped_upd

In [87]:
import subprocess
import sys

command = [
    sys.executable, "-m", "torch.distributed.run",
    "--standalone",
    f"--nproc_per_node={WORLD_SIZE}",
    str(RUNTIME / "model_d_fast_train.py"),
    "--config", str(ROOT / "fast_run_request.json"),
]

log_path = ROOT / "model_d_debug.log"

with log_path.open("w") as log:
    result = subprocess.run(
        command,
        cwd=str(RUNTIME),
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )

lines = log_path.read_text(errors="replace").splitlines()
print("Exit code:", result.returncode)
print("\n".join(lines[-80:]))

Exit code: 0

*****************************************
Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
*****************************************
RESUME_DEBUG {'script': '/kaggle/working/cs781-model-d-fast/runtime/model_d_fast_train.py', 'run_dir': '/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722', 'checkpoints': ['/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722/checkpoint-00003000.pt', '/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722/checkpoint-00003051.pt']}
RESUME_DEBUG {'script': '/kaggle/working/cs781-model-d-fast/runtime/model_d_fast_train.py', 'run_dir': '/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722', 'checkpoints': ['/kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722/checkpoint-00003000.pt', '/kaggle/working/cs781-model-d-fast/model_d_

In [88]:
import json
from pathlib import Path

request_path = ROOT / "fast_run_request.json"
request = json.loads(request_path.read_text())
actual_run_dir = Path(request["run_dir"])

print("Notebook RUN_DIR:", RUN_DIR)
print("Request file:", request_path)
print("Trainer run_dir:", actual_run_dir)
print("Paths match:", Path(RUN_DIR) == actual_run_dir)
print("Existing checkpoints:")
for p in sorted(actual_run_dir.glob("checkpoint-????????.pt")):
    print(p.name)

Notebook RUN_DIR: /kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722
Request file: /kaggle/working/cs781-model-d-fast/fast_run_request.json
Trainer run_dir: /kaggle/working/cs781-model-d-fast/model_d_fresh_20261011_095722
Paths match: True
Existing checkpoints:
checkpoint-00003000.pt
checkpoint-00003051.pt


## Model D checkpoint export


In [89]:
from IPython.display import display, FileLink

if RUN_TRAINING:
    if 'process' in globals() and process.poll() is None:
        raise RuntimeError('Trainer is still running. Wait for its DONE message before creating archives.')
    export, phase2_archive, resume_archive = export_checkpoints(ROOT, RUN_DIR,
                                                             build_resume=BUILD_RESUME_ARCHIVE)
    D_CHECKPOINT = Path(export['checkpoint'])
    display(FileLink(str(phase2_archive)))
    if BUILD_RESUME_ARCHIVE:
        display(FileLink(str(resume_archive)))
    print('For Phase2_EndToEnd.ipynb in this same session, set:')
    print('CHECKPOINT_OVERRIDES["D"] = ' + repr(str(D_CHECKPOINT)))
    print('For a new session, attach the saved Model D output and use its actual /kaggle/input/... .pt path.')
    print('Keep your original A, B and C checkpoint configuration unchanged.')
else:
    print('No real training/checkpoints exported during software-only inspection.')


Trained Model D checkpoint: /kaggle/working/cs781-model-d-fast/phase2_checkpoint/checkpoint-00003051.pt
Target reached: True | raw/scored tokens: 199950336 99975168
Phase 2 archive: /kaggle/working/model_d_phase2.zip | Resume archive: /kaggle/working/model_d_resume.zip


/kaggle/working/model_d_phase2.zip

/kaggle/working/model_d_resume.zip

For Phase2_EndToEnd.ipynb in this same session, set:
CHECKPOINT_OVERRIDES["D"] = '/kaggle/working/cs781-model-d-fast/phase2_checkpoint/checkpoint-00003051.pt'
For a new session, attach the saved Model D output and use its actual /kaggle/input/... .pt path.
Keep your original A, B and C checkpoint configuration unchanged.
